# Lyric Engine - Kaggle Training Notebook (T4 x2)

**Repo:** https://github.com/SMXFREEZE/lyric-engine  
**Branch:** `codex/emotional-specificity-module`  
**Runtime:** GPU T4 x2 (32 GB VRAM total) - Settings -> Accelerator -> GPU T4 x2

### Stages
1. Check GPU (multi-GPU)
2. Install dependencies
3. Clone / update repo from GitHub and checkout the metacognitive branch
4. Config (tokens, model, genre) - **fp16, no quantization needed on 32 GB**
5. Smoke validation (`python scripts/smoke_test.py`)
6. Scrape lyrics from Genius API
7. Build style vectors
8. **Viral DNA - scrape 55-country charts -> compute trend signal**
9. Verify pipeline
10. Load model + LoRA (fp16, device_map=auto)
11. Train (with viral conditioning vector)
12. Test generation + metacognitive outputs
13. **Generate full song (MusicGen large + Bark vocals)**
14. Save checkpoint

In [ ]:
# -- 1. Check GPU --
import torch

if not torch.cuda.is_available():
    print('NO GPU — go to Settings → Accelerator → GPU T4 x2')
    raise SystemExit('Enable GPU first')

n_gpus = torch.cuda.device_count()
total_vram = 0
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    total_vram += vram
    print(f'GPU {i}: {name}  ({vram:.1f} GB)')

print(f'\nTotal VRAM : {total_vram:.1f} GB')
print(f'GPUs       : {n_gpus}')

if total_vram >= 28:
    print('✓ T4 x2 — will use fp16, no quantization needed')
elif total_vram >= 14:
    print('⚠ Single T4 — will use 4-bit quantization')
else:
    print('⚠ Low VRAM — switching to 4-bit + small model')

!nvidia-smi

In [ ]:
# -- 2. Install dependencies --
import subprocess, sys


def pip_install(packages, label):
    print(f'Installing {label}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
    print(f'{label} installed.')


core_packages = [
    'transformers', 'peft', 'accelerate', 'bitsandbytes', 'trl',
    'datasets', 'lyricsgenius', 'billboard.py',
    'pronouncing', 'sentence-transformers', 'textblob',
    'fastapi', 'uvicorn', 'rich', 'typer', 'python-dotenv',
    'pydub', 'soundfile',
]
optional_audio_packages = ['audiocraft', 'suno-bark']

pip_install(core_packages, 'core dependencies')

audio_ready = True
try:
    pip_install(optional_audio_packages, 'audio dependencies')
except Exception as e:
    audio_ready = False
    print(f'Audio dependencies failed: {e}')
    print('Continuing without full audio generation support for this session.')

print(f'Audio ready: {audio_ready}')
print('Dependency setup complete.')

In [ ]:
# -- 3. Clone / update repo on the metacognitive branch --
import os, sys

REPO = 'https://github.com/SMXFREEZE/lyric-engine'
BRANCH = 'codex/emotional-specificity-module'
DEST = '/kaggle/working/lyric-engine'

if os.path.exists(f'{DEST}/.git'):
    print(f'Already cloned - fetching and switching to {BRANCH}...')
    !git -C {DEST} fetch origin
    !git -C {DEST} checkout {BRANCH}
    !git -C {DEST} pull origin {BRANCH}
else:
    print(f'Cloning repo on branch {BRANCH}...')
    !git clone -b {BRANCH} {REPO} {DEST}

%cd {DEST}
print()
print('Active branch:')
!git branch --show-current
print()
print('Latest commits:')
!git log --oneline -5

sys.path.insert(0, '.')

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/style_vectors', exist_ok=True)
print()
print(f'Checkpoints -> {CHECKPOINT_DIR}')

In [ ]:
# -- 4. Config --
# !! Add tokens as Kaggle Secrets (eye icon on left sidebar) when available !!
# Secret name: GENIUS_TOKEN       -> optional, improves Genius discovery/lookup
# Secret name: VAGALUME_API_KEY   -> optional, improves lyrics coverage
# Secret name: HF_TOKEN           -> optional, for HuggingFace dataset load/push

import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()


def maybe_secret(name: str) -> str:
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''


GENIUS_TOKEN = maybe_secret('GENIUS_TOKEN')
VAGALUME_API_KEY = maybe_secret('VAGALUME_API_KEY')
HF_TOKEN = maybe_secret('HF_TOKEN')

if GENIUS_TOKEN:
    os.environ['GENIUS_TOKEN'] = GENIUS_TOKEN
if VAGALUME_API_KEY:
    os.environ['VAGALUME_API_KEY'] = VAGALUME_API_KEY
if HF_TOKEN:
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# ?? Auto-detect best settings based on available VRAM ????????????????????????
import torch
total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory / 1e9
    for i in range(torch.cuda.device_count())
)

# T4 x2 = 32 GB ? fp16, no quantization, larger batches, bigger models
# Single T4   = 16 GB ? 4-bit, smaller batches
USE_4BIT       = total_vram < 20          # False on T4 x2
MUSICGEN_SIZE  = 'large' if total_vram >= 20 else 'small'
LORA_RANK      = 64  if total_vram >= 20 else 16     # bigger adapter on more VRAM
BATCH_SIZE     = 4   if total_vram >= 20 else 2
GRAD_ACCUM     = 8   if total_vram >= 20 else 16     # eff batch = 32 either way

# ?? Model ?????????????????????????????????????????????????????????????????????
# Mistral 7B fp16  ? 14 GB ? fits comfortably in 32 GB leaving room for LoRA + activations
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'

# ?? Training config ???????????????????????????????????????????????????????????
GENRE      = 'trap'   # trap / rnb / indie / pop / drill / hip_hop / country / latin / afrobeats / kpop
EPOCHS     = 3        # more epochs on T4 x2 since we have the power
LR         = 2e-4

# ?? Scraper config ????????????????????????????????????????????????????????????
MAX_SONGS_PER_ARTIST = 50

# ?? Chart config ??????????????????????????????????????????????????????????????
SCRAPE_CHARTS  = True
HF_DATASET_REPO = 'SMXFREEZE/lyric-engine-data'

print(f'Base model    : {BASE_MODEL}')
print(f'Genre         : {GENRE}')
print(f'Total VRAM    : {total_vram:.1f} GB')
print(f'4-bit quant   : {USE_4BIT}')
print(f'MusicGen size : {MUSICGEN_SIZE}')
print(f'LoRA rank     : {LORA_RANK}')
print(f'Eff. batch    : {BATCH_SIZE * GRAD_ACCUM}')
print(f'Epochs        : {EPOCHS}')
print(f'Genius token  : {"?" if GENIUS_TOKEN else "- optional"}')
print(f'Vagalume key  : {"?" if VAGALUME_API_KEY else "- optional"}')
print(f'HF token      : {"?" if HF_TOKEN else "- optional unless pushing/loading HF"}')


In [ ]:
# -- 5. Phase A smoke validation --
%cd {DEST}

import subprocess, sys

print('Running early smoke test on the metacognitive branch...')
subprocess.check_call([sys.executable, 'scripts/smoke_test.py'])

print()
print('Phase A infrastructure validation OK')

In [ ]:
# -- 6. Acquire lyrics dataset: Kaggle upload -> HF repo -> multi-provider live scrape --
#
# Kaggle is often blocked by live lyrics sites. So the order is:
#   1. Prefer a Kaggle-uploaded JSONL dataset if present
#   2. Fall back to HuggingFace dataset repo if it exists
#   3. Only then try a live multi-provider scrape
#
# Expected JSONL schema per row:
#   {"artist": "...", "title": "...", "genre": "trap", "lyrics": "..."}

import json, os, shutil

out_path = f'data/raw/{GENRE}.jsonl'
os.makedirs('data/raw', exist_ok=True)
existing_titles = set()
songs = []

# -- 0. Prefer a Kaggle-uploaded dataset -------------------------------------
input_candidates = [
    f'/kaggle/input/lyric-engine-data/{GENRE}.jsonl',
    f'/kaggle/input/lyric-engine-data/data/raw/{GENRE}.jsonl',
    f'/kaggle/input/lyrics-data/{GENRE}.jsonl',
    f'/kaggle/input/lyrics-data/data/raw/{GENRE}.jsonl',
]
local_input = next((p for p in input_candidates if os.path.exists(p)), None)

if local_input:
    shutil.copyfile(local_input, out_path)
    with open(out_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            existing_titles.add(row.get('title', ''))
    print(f'Loaded local Kaggle dataset: {local_input}')
    print(f'Rows available: {len(existing_titles)}')

# -- 1. HuggingFace fallback --------------------------------------------------
elif HF_TOKEN:
    try:
        from datasets import load_dataset
        print(f'Loading existing dataset from HuggingFace: {HF_DATASET_REPO}...')
        existing_ds = load_dataset(HF_DATASET_REPO, split='train', token=HF_TOKEN)
        genre_rows = [r for r in existing_ds if r.get('genre') == GENRE]
        existing_titles = {r['title'] for r in genre_rows}
        with open(out_path, 'w', encoding='utf-8') as f:
            for r in genre_rows:
                f.write(json.dumps(r, ensure_ascii=False) + '
')
        print(f'Loaded {len(genre_rows)} existing {GENRE} songs from HF')
    except Exception as e:
        print(f'HF load skipped ({e}) -> trying live multi-provider scrape...')

# -- 2. Live multi-provider scrape only if still empty -----------------------
if not existing_titles:
    from src.data.scraper import run_scrape

    print('No uploaded dataset found ? running live multi-provider scrape...')
    print('Discovery: Deezer top tracks + Genius fallback')
    print('Lyrics: Genius direct -> Vagalume -> LRCLIB -> lyrics.ovh -> Genius search')
    records = run_scrape(
        genres=[GENRE],
        max_songs_per_artist=MAX_SONGS_PER_ARTIST,
        out_dir='data/raw',
    )
    songs = [r for r in records if r.get('genre') == GENRE]
    existing_titles = {r.get('title', '') for r in songs}
    print(f'Live scrape collected {len(songs)} rows for {GENRE}')

# -- Final data check ---------------------------------------------------------
row_count = 0
if os.path.exists(out_path):
    with open(out_path, 'r', encoding='utf-8') as f:
        row_count = sum(1 for line in f if line.strip())

print(f'
Rows available for training: {row_count}')
if row_count == 0:
    raise SystemExit(
        'No lyrics data available. Upload /kaggle/input/lyric-engine-data/{GENRE}.jsonl, '
        'create the HF repo first, or add VAGALUME_API_KEY / GENIUS_TOKEN and retry.'
    )

# -- Optional push back to HuggingFace if this run scraped new songs ----------
if len(songs) > 0 and HF_TOKEN:
    try:
        from datasets import load_dataset, Dataset
        import pandas as pd

        print(f'
Pushing {len(songs)} new songs to HuggingFace: {HF_DATASET_REPO}...')
        try:
            existing_ds = load_dataset(HF_DATASET_REPO, split='train', token=HF_TOKEN)
            df_existing = existing_ds.to_pandas()
        except Exception:
            df_existing = pd.DataFrame()

        df_new = pd.DataFrame(songs)
        df_full = pd.concat([df_existing, df_new], ignore_index=True).drop_duplicates(subset=['artist', 'title'])
        Dataset.from_pandas(df_full).push_to_hub(HF_DATASET_REPO, token=HF_TOKEN)
        print(f'Pushed -> HF dataset now has {len(df_full)} songs total')
    except Exception as e:
        print(f'HF push failed: {e} -> songs remain saved locally at {out_path}')


In [ ]:
# -- 7. Build style vectors --
from src.data.style_extractor import build_style_vectors

print('Building artist style vectors...')
style_vectors = build_style_vectors(
    jsonl_path=f'data/raw/{GENRE}.jsonl',
    out_dir='data/style_vectors',
    min_songs=2,
)
print(f'Style vectors built for {len(style_vectors)} artists')

In [ ]:
# -- 8. Viral DNA — scrape 55-country charts + compute trend conditioning vector --
import json
import numpy as np

VIRAL_VECTOR_PATH = 'data/viral_conditioning.npy'
VIRAL_REPORT_PATH = 'data/viral_report.json'

if SCRAPE_CHARTS:
    print('Scraping charts from Billboard / Deezer / Spotify CSV / iTunes...')
    print('(Covers 55+ countries including Morocco, all of Africa, Asia, LatAm, Europe)\n')

    try:
        from src.data.chart_scraper import run_weekly, compute_viral_scores
        from src.data.viral_analyzer import run_analysis, build_conditioning_vector

        # 1. Fetch charts (this hits free public APIs — no login needed)
        chart_data = run_weekly(output_dir='data/charts')
        print(f'  Charts fetched: {len(chart_data)} country/source entries')

        # 2. Compute viral scores (songs charting in many countries = viral DNA)
        viral_scores = compute_viral_scores(chart_data)
        top_viral = sorted(viral_scores.items(), key=lambda x: x[1], reverse=True)[:20]
        print('\nTop 20 globally viral tracks:')
        for title, score in top_viral:
            print(f'  [{score:5.1f}] {title}')

        # 3. Run full viral analysis (region profiles, trend velocity, universal DNA)
        report = run_analysis(chart_data, output_dir='data/viral_analysis')

        # 4. Build 32-dim conditioning vector for model training
        viral_vec = build_conditioning_vector(report)
        np.save(VIRAL_VECTOR_PATH, viral_vec)

        with open(VIRAL_REPORT_PATH, 'w') as f:
            json.dump(report, f, indent=2, default=str)

        print(f'\nViral conditioning vector shape : {viral_vec.shape}')
        print(f'Saved to: {VIRAL_VECTOR_PATH}')
        print('\nRegion DNA profiles:')
        for region, profile in report.get('region_profiles', {}).items():
            print(f'  {region:15s}: BPM={profile.get("avg_bpm", "?"):.0f}  energy={profile.get("avg_energy", "?"):.2f}')

    except Exception as e:
        print(f'Chart scraping error: {e}')
        print('Falling back to neutral conditioning vector...')
        viral_vec = np.zeros(32, dtype=np.float32)
        np.save(VIRAL_VECTOR_PATH, viral_vec)

else:
    print('SCRAPE_CHARTS=False — loading saved vector or using neutral...')
    if os.path.exists(VIRAL_VECTOR_PATH):
        viral_vec = np.load(VIRAL_VECTOR_PATH)
        print(f'Loaded: {VIRAL_VECTOR_PATH}')
    else:
        viral_vec = np.zeros(32, dtype=np.float32)
        np.save(VIRAL_VECTOR_PATH, viral_vec)
        print('Using neutral zero vector')

print(f'\nViral conditioning vector (first 8 dims): {viral_vec[:8].round(3)}')
print('Viral DNA ready ✓')

In [ ]:
# -- 9. Verify pipeline --
import json
from src.data.phoneme_annotator import annotate_line
from src.data.rhyme_labeler import detect_scheme
from src.data.valence_scorer import score_lyrics

with open(f'data/raw/{GENRE}.jsonl') as f:
    sample = json.loads(f.readline())

lines = sample['lyrics'].splitlines()[:8]
lyrics_block = chr(10).join(lines)
print(f"Artist : {sample['artist']}")
print(f"Title  : {sample['title']}")
print()

scheme = detect_scheme(lines)
print(f"Rhyme scheme  : {scheme['scheme_str']} ({scheme['scheme_type']})")
print(f"Rhyme density : {scheme['rhyme_density']}")
print()

for em in score_lyrics(lyrics_block):
    ann = annotate_line(em.text)
    print(f"  [{em.valence:+.2f}v {em.arousal:.2f}a] [{ann.total_syllables:2d}syl] {em.text[:60]}")

print()
print('Pipeline OK')

In [ ]:
# -- 10. Load base model + LoRA (fp16 on T4 x2, 4-bit fallback on single GPU) --
import torch
from src.model.lyrics_model import load_base_model, apply_lora, LyricsModel

print(f'Loading {BASE_MODEL}...')
print(f'Mode: {"4-bit quantization" if USE_4BIT else "fp16 full precision"}')
print(f'device_map=auto → will spread across {torch.cuda.device_count()} GPU(s)\n')

base, tokenizer = load_base_model(
    BASE_MODEL,
    use_4bit=USE_4BIT,
    device=None,        # let device_map='auto' handle distribution
)

# Apply LoRA adapter
base = apply_lora(base, rank=LORA_RANK, alpha=LORA_RANK * 2)

# Wrap with phonetic head
model = LyricsModel(base, d_model=base.config.hidden_size)

base.print_trainable_parameters()

# Show which layers are on which GPU
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {used:.1f}/{total:.1f} GB used')

print('\nModel ready ✓')

In [ ]:
# -- 11. Train genre LoRA (with viral DNA conditioning) --
import numpy as np
from src.training.sft import train_sft

# Load the viral conditioning vector built in cell 8
viral_vec = np.load(VIRAL_VECTOR_PATH) if os.path.exists(VIRAL_VECTOR_PATH) else np.zeros(32, dtype=np.float32)
print(f'Viral conditioning vector loaded: {viral_vec.shape}, norm={np.linalg.norm(viral_vec):.3f}')

# Stage 1: General foundation (high rank, all data) — skipped if checkpoint exists
stage1_ckpt = f'{CHECKPOINT_DIR}/stage1_general'
if not os.path.exists(stage1_ckpt):
    print('\n[Stage 1] Training general foundation LoRA (rank 64)...')
    train_sft(
        stage=1,
        genre=GENRE,
        data_path=f'data/raw/{GENRE}.jsonl',
        base_model=BASE_MODEL,
        output_dir=CHECKPOINT_DIR,
        batch_size=BATCH_SIZE,
        grad_accum_steps=GRAD_ACCUM,
        epochs=1,
        lr=LR,
        lora_rank=64,
        alpha=128,
        use_4bit=USE_4BIT,
        style_vec_dir='data/style_vectors',
        viral_conditioning=viral_vec,
    )
    print('Stage 1 done ✓')
else:
    print(f'Stage 1 checkpoint found at {stage1_ckpt} — skipping')

# Stage 2: Genre-specific LoRA (smaller rank, fine-tuned per genre)
print(f'\n[Stage 2] Training {GENRE} genre LoRA (rank {LORA_RANK})...')
train_sft(
    stage=2,
    genre=GENRE,
    data_path=f'data/raw/{GENRE}.jsonl',
    base_model=BASE_MODEL,
    output_dir=CHECKPOINT_DIR,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    epochs=EPOCHS,
    lr=LR,
    lora_rank=LORA_RANK,
    alpha=LORA_RANK * 2,
    use_4bit=USE_4BIT,
    style_vec_dir='data/style_vectors',
    viral_conditioning=viral_vec,   # ← viral DNA wired in as training signal
)
print(f'\nTraining complete ✓ — checkpoint: {CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}')

In [ ]:
# -- 12. Test generation + metacognitive outputs --
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTok
from peft import PeftModel
from src.inference.engine import LyricsEngine, SongMemory

ckpt_path = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading checkpoint from {ckpt_path}...')
tok = HFTok.from_pretrained(ckpt_path)
tok.pad_token = tok.eos_token

base_load_kwargs = {'device_map': 'auto'}
if USE_4BIT:
    base_load_kwargs['load_in_4bit'] = True
else:
    base_load_kwargs['torch_dtype'] = torch.float16

base_mdl = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **base_load_kwargs)
trained_mdl = PeftModel.from_pretrained(base_mdl, ckpt_path)

engine = LyricsEngine(trained_mdl, tok, device=device, beam_size=8)
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print()
print(f'=== Generated {GENRE.upper()} verse ===')
verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')
for i, line in enumerate(verse, 1):
    print(f'  {i}. {line}')

probe = engine.generate_line(memory, target_arc_valence=0.1, target_arc_arousal=0.6, top_n=1, section='VERSE')[0]
print()
print('=== Workspace candidate fields ===')
print(f'  workspace_score      : {probe.workspace_score:.3f}')
print(f'  workspace_confidence : {probe.workspace_confidence:.3f}')
print(f'  workspace_conflict   : {probe.workspace_conflict:.3f}')
print(f'  decision_mode        : {probe.decision_mode}')
print(f'  decision_trace       : {list(probe.decision_trace)}')

analysis = engine.analyze_song(memory)
print()
print('=== Metacognitive analysis ===')
print(f"  mode_counts    : {analysis['metacognition']['mode_counts']}")
print(f"  avg_confidence : {analysis['metacognition']['avg_confidence']}")
print(f"  avg_conflict   : {analysis['metacognition']['avg_conflict']}")
print(f"  last_workspace : {'present' if analysis['last_workspace'] else 'missing'}")

assert hasattr(probe, 'workspace_score')
assert hasattr(probe, 'workspace_confidence')
assert hasattr(probe, 'workspace_conflict')
assert hasattr(probe, 'decision_mode')
assert hasattr(probe, 'decision_trace')
assert 'metacognition' in analysis
assert 'last_workspace' in analysis
assert analysis['last_workspace'] is not None

print()
print('Phase B model validation OK')

In [ ]:
# -- 13. Generate full song (MusicGen large + Bark vocals + final MP3) --
# Uses the lyrics generated in cell 12 above + the trained genre style

if not globals().get('audio_ready', True):
    print('Skipping audio generation because optional audio dependencies were not installed in cell 2.')
    print('Re-run cell 2 after fixing audio packages if you want full instrumental + vocal output.')
else:
    from src.audio.song_assembler import SongAssembler, SongSpec

    # Grab generated verse from test cell above (or define custom)
    # 'verse' is the list of lines generated in cell 12
    generated_lyrics = {
        'verse1':  verse[:4] if len(verse) >= 4 else verse,
        'chorus':  verse[4:8] if len(verse) >= 8 else [
            'Turn up the heat, we burning bright tonight',
            'Everything I touch turns gold in the light',
        ],
        'verse2':  verse[:4] if len(verse) >= 4 else verse,
        'chorus2': verse[4:8] if len(verse) >= 8 else [
            'Turn up the heat, we burning bright tonight',
            'Everything I touch turns gold in the light',
        ],
    }

    spec = SongSpec(
        title=f'AI_{GENRE.upper()}_001',
        genre=GENRE,
        bpm=140 if GENRE in ('trap', 'drill') else 120,
        mood='dark' if GENRE in ('trap', 'drill', 'alt_emo') else 'hype',
        language='en',
        voice_idx=0,
        lyrics=generated_lyrics,
        sections=[
            ('intro',   8),
            ('verse1',  16),
            ('chorus',  12),
            ('verse2',  16),
            ('chorus2', 12),
            ('bridge',  8),
            ('outro',   8),
        ],
        output_dir='/kaggle/working/songs',
    )

    print(f'Assembling song: {spec.title}')
    print(f'MusicGen size : {MUSICGEN_SIZE}  (large = best quality, uses ~6 GB VRAM)')
    print(f'Bark          : small model (vocal synthesis, ~2 GB VRAM)')
    print(f'Total sections: {[s[0] for s in spec.sections]}')
    print()

    # Free LLM VRAM before loading audio models
    import gc
    try:
        del base, model
        gc.collect()
        torch.cuda.empty_cache()
        print('LLM unloaded to free VRAM for audio models')
    except Exception:
        pass

    # Show VRAM state before audio generation
    for i in range(torch.cuda.device_count()):
        used = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {used:.1f}/{total:.1f} GB free after LLM unload')

    assembler = SongAssembler(
        musicgen_size=MUSICGEN_SIZE,
        bark_small=True,
        device='auto',
    )

    song = assembler.generate_song(spec, skip_vocals=False, skip_instrumental=False)

    print(f'\n{"="*60}')
    print(f'Song generated successfully!')
    print(f'Output: {song.output_path}')
    print(f'Duration: {len(song.mixed) / 44100:.1f}s')
    print(f'\nFiles in /kaggle/working/songs/:')
    import os, pathlib
    for p in sorted(pathlib.Path('/kaggle/working/songs').rglob('*')):
        if p.is_file():
            size = p.stat().st_size / 1e6
            print(f'  {p.name:50s} {size:.1f} MB')
    print('\nDownload from: Kaggle -> Output tab (right sidebar) -> songs/')

In [ ]:
# -- 14. Checkpoint summary --
import os

ckpt = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_{EPOCHS}'
print(f'Checkpoint: {ckpt}')
print()
if os.path.exists(ckpt):
    total_mb = 0
    for fname in sorted(os.listdir(ckpt)):
        size = os.path.getsize(f'{ckpt}/{fname}')
        total_mb += size / 1e6
        print(f'  {fname:45s} {size/1e6:6.1f} MB')
    print(f'\n  Total: {total_mb:.0f} MB')
else:
    print('Checkpoint directory not found — did training complete?')

print()
print('To download: Kaggle → Output tab (right sidebar) → Download')